In [81]:
# Author: Alyssa Feger
# 4/07/26

import pandas as pd
from recipe import transform_list

def raw_materials(materials_list: list[str]):
    materials = transform_list(materials_list)  # Makes the list into a dictionary
    recipe_df = pd.read_csv("recipes.tsv", sep="\t")  # Makes a dataframe out of the recipe data
    raw_dict = raw_materials_helper(materials, recipe_df)  

    # Makes the return dictionary into a easy to read string
    change_str = ""
    for mat in raw_dict.keys():
        material = mat[10:]
        material_split = material.split("_")  # If mat is two words it splits on the _ and then joins them by a space
        material = " ".join(material_split)
        string = f"{raw_dict[mat]} {material}, "
        change_str = change_str + string

    change_str = change_str[:-2]  # Gets rid of last comma and space
    
    return change_str

    
def raw_materials_helper(materials_dict: dict[str, int], recipe_df): #should probably add another parameter
    mat_dict: dict[str, int] = {}
    
    for material in materials_dict.keys():
        thing_dict: dict[str, int] = {}  # Dictionary for material's recipe
        materials_mask = recipe_df.loc[:, "Craftable"] == material 
        material_df = recipe_df.loc[materials_mask, :]  # Creates a dataframe specifically for needed material
        
        if material_df.empty:  # iF material_df is empty then material was not found in dataframe
            print(f"{material} does not exist in recipe list")
        else:
            index = list(material_df.index)[0]  # Finding index of material in material_df to use indexing
            material_num = materials_dict[material]
            count = int(material_df.loc[index, "Count"])
            recipe = material_df.loc[index, "Recipe"]


            # If number of material cannot be evenly divided it does integer division and adds one
            # So that needed raw materials are not under represented
            if material_num % count != 0:
                num_count = (material_num // count) + 1
            else:
                num_count = material_num // count

            
            if recipe == "0":  # Base case for raw materials
                if material in mat_dict.keys(): 
                    mat_dict[material] = mat_dict[material] + materials_dict[material]  # If item in mat_dict then count is added to count already in
                else:
                    mat_dict[material] = materials_dict[material]
            else:
                recipe_list = recipe.split(",")  # Splits the recipe list to get each item
                for thing in recipe_list:
                    thing_split = thing.split()  # Splits the item to get its number and material name
                    mat_count = int(thing_split[0]) * num_count  
                    thing_dict[thing_split[1]] = mat_count  # Adds item and it's count to thing_dict

                raw_dict = raw_materials_helper(thing_dict, recipe_df)  # Recursively calls function on given material's recipe items until base case is called

                
                # Goes through raw_dict and see if it exists in mat_dict already
                for raw in raw_dict.keys():
                    if raw in mat_dict.keys():
                        mat_dict[raw] = mat_dict[raw] + raw_dict[raw]  # If item in mat_dict then count is added to count already in
                    else:
                        mat_dict[raw] = raw_dict[raw]  # If item not in mat_dict then raw material is added to mat_dict 

    return mat_dict




print(raw_materials(["27 orange concrete"]))
print(raw_materials(["4 cartography table", "60 ladder"]))

4 orange tulip, 16 sand, 16 gravel
9 sugar cane, 22 oak log
